In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime
import os
import warnings

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import pennylane as qml

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

data = pd.read_csv('../data/sp500_dataset.csv', index_col='Date', parse_dates=True)

data['Target'] = data['Close'].shift(-1)
data = data.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume', 
            'SMA_50', 'SMA_200', 'MACD', 'Signal', 'RSI', 'Momentum']

X = data[features].values
y = data['Target'].values
dates = data.index.to_numpy()

LOOKBACK_WINDOW = 20

def create_sequences_with_dates(X, y, dates, window_size):
    X_seq, y_seq, y_dates = [], [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:(i + window_size)])
        y_seq.append(y[i + window_size])
        y_dates.append(dates[i + window_size])
    return (np.array(X_seq, dtype=np.float32),
            np.array(y_seq, dtype=np.float32),
            np.array(y_dates, dtype='datetime64[ns]'))

X_seq, y_seq, dates_seq = create_sequences_with_dates(X, y, dates, LOOKBACK_WINDOW)

TEST_SIZE = 0.20
test_split_index = int(len(X_seq) * (1 - TEST_SIZE))
X_train_raw, X_test_raw = X_seq[:test_split_index], X_seq[test_split_index:]
y_train_raw, y_test_raw = y_seq[:test_split_index], y_seq[test_split_index:]
dates_test = dates_seq[test_split_index:]

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

n_features = X_train_raw.shape[2]
X_train_2d = X_train_raw.reshape(-1, n_features)
X_test_2d = X_test_raw.reshape(-1, n_features)

scaler_X.fit(X_train_2d)
X_dev_scaled = scaler_X.transform(X_train_2d).reshape(X_train_raw.shape)
X_test_scaled = scaler_X.transform(X_test_2d).reshape(X_test_raw.shape)

scaler_y.fit(y_train_raw.reshape(-1, 1))
y_dev_scaled = scaler_y.transform(y_train_raw.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test_raw.reshape(-1, 1)).flatten()

val_fraction = 0.20
val_index = int(len(X_dev_scaled) * (1 - val_fraction))
X_train_scaled, X_val_scaled = X_dev_scaled[:val_index], X_dev_scaled[val_index:]
y_train_scaled, y_val_scaled = y_dev_scaled[:val_index], y_dev_scaled[val_index:]

print("Dataset shapes after preparation:")
print(f"  X_train: {X_train_scaled.shape}, y_train: {y_train_scaled.shape}")
print(f"  X_val: {X_val_scaled.shape}, y_val: {y_val_scaled.shape}")
print(f"  X_test: {X_test_scaled.shape}, y_test: {y_test_scaled.shape}")
print(f"  Test dates: {dates_test.shape}")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).view(-1, 1)

batch_size = 16
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

n_qubits = 4
n_qlayers = 2

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

class QGRUCell(nn.Module):
    def __init__(self, n_qubits, n_qlayers, hidden_size):
        super(QGRUCell, self).__init__()
        self.n_qubits = n_qubits
        self.n_qlayers = n_qlayers
        self.hidden_size = hidden_size
        weight_shapes = {"weights": (n_qlayers, n_qubits)}
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        self.update_gate = nn.Linear(n_qubits, hidden_size)
        self.reset_gate = nn.Linear(n_qubits, hidden_size)
        self.hidden_gate = nn.Linear(hidden_size + n_qubits, hidden_size)

    def forward(self, x, h_prev):
        q_out = self.qlayer(x)
        z_t = torch.sigmoid(self.update_gate(q_out))
        r_t = torch.sigmoid(self.reset_gate(q_out))
        combined = torch.cat([r_t * h_prev, q_out], dim=1)
        h_tilde = torch.tanh(self.hidden_gate(combined))
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        return h_t

class QGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, n_qubits, n_qlayers, output_size=1, dropout=0.3):
        super(QGRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.n_qubits = n_qubits
        self.input_projection = nn.Linear(input_size, n_qubits)
        self.qgru_cell = QGRUCell(n_qubits, n_qlayers, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.dropout2 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h_t = torch.zeros(batch_size, self.hidden_size, device=x.device)
        for t in range(seq_len):
            x_t = x[:, t, :]
            q_in = torch.tanh(self.input_projection(x_t))
            h_t = self.qgru_cell(q_in, h_t)
            
        out = self.bn1(h_t)
        out = self.dropout1(out)
        out = self.relu(self.fc1(out))
        out = self.bn2(out)
        out = self.dropout2(out)
        out = self.fc2(out)
        return out

model = QGRUModel(input_size=11, hidden_size=64, n_qubits=n_qubits, n_qlayers=n_qlayers, dropout=0.3)
criterion = nn.MSELoss()
optimizer = optim.NAdam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20, min_lr=1e-6)

EPOCHS = 400
patience_early = 100
os.makedirs('models', exist_ok=True)
best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0

train_losses, val_losses = [], []
train_maes, val_maes = [], []

for epoch in range(EPOCHS):
    model.train()
    total_train_loss, total_train_mae = 0.0, 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
        total_train_mae += torch.abs(outputs - batch_y).mean().item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_mae = total_train_mae / len(train_loader)
    train_losses.append(avg_train_loss)
    train_maes.append(avg_train_mae)
    
    model.eval()
    total_val_loss, total_val_mae = 0.0, 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            val_loss = criterion(outputs, batch_y)
            total_val_loss += val_loss.item()
            total_val_mae += torch.abs(outputs - batch_y).mean().item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_mae = total_val_mae / len(val_loader)
    val_losses.append(avg_val_loss)
    val_maes.append(avg_val_mae)
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), 'models/QGRU_regression.pth')
        print(f"Epoch {epoch+1}/{EPOCHS} - loss: {avg_train_loss:.6f} - val_loss: {avg_val_loss:.6f} - Saved model")
    else:
        patience_counter += 1
        if patience_counter >= patience_early:
            print(f"Early stopping at epoch {epoch+1}. Best val_loss: {best_val_loss:.6f}")
            break

model.load_state_dict(torch.load('../models/QGRU_regression.pth'))
model.eval()
predictions_scaled, actuals_scaled = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        predictions_scaled.extend(outputs.numpy())
        actuals_scaled.extend(batch_y.numpy())

y_pred_scaled = np.array(predictions_scaled).flatten()
y_test_scaled = np.array(actuals_scaled).flatten()
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_original = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1)).flatten()

mse = mean_squared_error(y_test_original, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)
mape = np.mean(np.abs((y_test_original - y_pred) / y_test_original)) * 100

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"R2 Score: {r2:.4f} ({r2*100:.2f}%)")
